# 04 — Results visualisation

The answer to the research question, and the figures for the write-up.

> How does model scale influence the effectiveness of joint versus sequential pruning and
> quantisation in decoder-only language models?

Read the numbers in this order, and stop early if a check fails:

1. Are the joint/sequential pairs complete at every scale and budget?
2. Were the optimisation budgets matched?
3. Is the joint gain larger than the seed-to-seed spread? **If not, the honest answer is that this
   experiment cannot distinguish the two pipelines.**
4. Does the gain trend with scale?
5. Does the Qwen point agree in sign with the interpolated Pythia trend?

For the final figures, prefer `python scripts/generate_plots.py` — it writes the same figures
reproducibly.

In [ ]:
from scale_aware_compression.experiments.runner import ExperimentTracker
from scale_aware_compression.logging_utils import configure_logging

configure_logging("INFO")
records = ExperimentTracker("../outputs/metrics").load_all()
print(f"{len(records)} record(s)")

by_method: dict[str, int] = {}
for record in records:
    method = str(record.get("compression_method"))
    by_method[method] = by_method.get(method, 0) + 1
print(by_method)

In [ ]:
from scale_aware_compression.config import load_config
from scale_aware_compression.experiments.scale_sweep import build_sweep_plan, find_comparison_pairs

# Which comparisons the plan can support at all. A missing pair means a whole point on Figure 1
# is absent.
config = load_config("../configs/experiments/main_scale_sweep.yaml")
plan = build_sweep_plan(config)
pairs = find_comparison_pairs(plan)
print(f"{plan.num_runs} planned run(s), {len(pairs)} joint/sequential pair(s)")
for sequential, joint in pairs:
    print(f"  {sequential.model_name:14s} {sequential.budget_label:11s} seed={sequential.seed}")

In [ ]:
from scale_aware_compression.metrics.joint_gain import joint_gain_summary
from scale_aware_compression.models.registry import get_model_spec
from scale_aware_compression.visualisation.tables import rows_to_markdown

# Joint gain per (model, budget, seed) from the recorded retention scores.
scores: dict[tuple[str, str, int], dict[str, float]] = {}
for record in records:
    method = record.get("compression_method")
    if method not in {"sequential", "joint"}:
        continue
    retention = record.get("quality", {}).get("retention", {}).get("perplexity_retention")
    if retention is None:
        continue
    key = (record["model_name"], record.get("budget_label", ""), record.get("seed", 0))
    scores.setdefault(key, {})[method] = retention

rows = []
for (model, budget, seed), arms in sorted(scores.items()):
    if "sequential" not in arms or "joint" not in arms:
        continue
    spec = get_model_spec(model)
    summary = joint_gain_summary(
        model_name=model,
        size_label=spec.size_label,
        parameter_count=spec.parameter_count,
        budget_label=budget,
        metric_name="perplexity_retention",
        joint_score=arms["joint"],
        sequential_score=arms["sequential"],
        seed=seed,
    )
    rows.append(summary.to_dict())

print(rows_to_markdown(rows) if rows else "No matched joint/sequential pairs recorded yet.")

In [ ]:
import statistics as stats

# The credibility check: a gain smaller than the seed spread is not a finding.
grouped: dict[tuple[str, str], list[float]] = {}
for row in rows:
    grouped.setdefault((str(row["model_name"]), str(row["budget_label"])), []).append(
        float(row["absolute_gain"])
    )

for (model, budget), gains in sorted(grouped.items()):
    mean = stats.fmean(gains)
    spread = stats.stdev(gains) if len(gains) > 1 else float("nan")
    verdict = (
        "conclusive" if len(gains) > 1 and abs(mean) > spread else "WITHIN NOISE / too few seeds"
    )
    print(f"{model:14s} {budget:11s} n={len(gains)} gain={mean:+.3f} spread={spread:.3f}  {verdict}")

## Still to do

Once `visualisation/plots.py` and `experiments/scale_sweep.scale_trend` are implemented:

- **Figure 1** joint gain vs parameter count, log x, one line per budget, error bars from the seed
  spread, horizontal line at zero, Qwen marked separately and excluded from any fit
- **Figure 2** measured latency vs sparsity with the `1/(1-sparsity)` bound overlaid
- **Figure 3** retention vs checkpoint size, one panel per model
- **Figure 4** joint gain vs training-cost overhead
- **Table 7** the Qwen transfer assessment via `experiments.validation.assess_transfer`